# Synthetic Data Generation

## 1. Config

In [ ]:
print("===========================| Config started... |===\n")

CONFIG_FILE = "qwen7b_lora_10.json"

print("\n===========================| Config completed. |===")

## 2. Setup

In [ ]:
print("===========================| Setup started... |===\n")

import os
import subprocess

def select_gpu():
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.free", "--format=csv,nounits,noheader"],
        capture_output=True, text=True
    )
    free_memories = [int(x) for x in result.stdout.strip().split("\n")]
    best = free_memories.index(max(free_memories))
    print(f"Selected GPU {best} ({max(free_memories)} MiB free)")
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best)

select_gpu()

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import random
import numpy as np
import json
import torch
from pathlib import Path
import pandas as pd
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
)
from peft import PeftModel, LoraConfig, get_peft_model, TaskType
from datasets import Dataset, load_dataset
from opacus import PrivacyEngine
from torch.utils.data import DataLoader

import warnings
warnings.filterwarnings("ignore", message="Secure RNG turned off")
warnings.filterwarnings("ignore", message="Full backward hook is firing")
warnings.filterwarnings("ignore", message="First batch is empty")

def set_all_seeds(n):
    os.environ["PYTHONHASHSEED"] = str(n)
    random.seed(n)
    np.random.seed(n)
    torch.manual_seed(n)
    torch.cuda.manual_seed_all(n)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def get_config():
    config = json.loads(Path(CONFIG_FILE).read_text(encoding="utf-8"))
    name = Path(CONFIG_FILE).stem
    seed = config["seed"]
    config["name"] = name
    config["run_name"] = f"{name}_{seed}"
    config["start_idx"] = seed * config["num_samples"]
    if isinstance(config.get("dataset"), dict):
        config["dataset"] = config["dataset"]
    print(f"Experiment: {config['run_name']}")
    print(f"- Strategy: {config['strategy']}")
    print(f"- Model: {config['model']}")
    print(f"- Samples: {config['start_idx']} → {config['start_idx'] + config['num_samples'] - 1}")
    return config

config = get_config()

if not torch.cuda.is_available():
    raise SystemExit("No GPU available. Exiting...")

set_all_seeds(config["seed"])

Path(config["output_dir"]).mkdir(exist_ok=True)

print("\n===========================| Setup completed. |===")

## 3. Load Model

In [ ]:
print("===========================| Load Model started... |===\n")

def load_model_and_tokenizer(model_name):
    print(f"Loading {model_name}...")
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"
    
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto",
        trust_remote_code=True,
    )
    
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer(config["model"])

print("\n===========================| Load Model completed. |===")

## 4. Load Data

In [ ]:
print("===========================| Load Data started... |===\n")

OUTPUT_DIR = Path(config["dataset_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = config["dataset_name"]
SEED = config["dataset_seed"]

datasets_config_meta = []
for entry in config["datasets"]:
    sample = entry["sample"]
    if sample is None:
        frac, n = None, None
    elif isinstance(sample, float):
        frac, n = sample, None
    else:
        frac, n = None, sample
    datasets_config_meta.append((entry["file"], entry["description"], frac, n))

def ensure_datasets(output_dir, dataset_name, datasets_meta, seed):
    if all((output_dir / filename).exists() for filename, *_ in datasets_meta):
        print("All dataset files already exist, skipping download.\n")
        return

    print("Downloading dataset...")
    train_df = load_dataset(dataset_name, split="train").to_pandas()

    train_df = train_df.drop(columns=config.get("drop_columns", []))

    for col in config.get("str_fillna_columns", []):
        train_df[col] = train_df[col].fillna("").astype(str)
    
    for col in config.get("int_to_str_columns", []):
        train_df[col] = train_df[col].apply(lambda x: "" if pd.isna(x) else str(int(x)))
    
    for col in config.get("str_to_int_columns", []):
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce").astype("Int64")
    
    for col in config.get("str_to_float_columns", []):
        train_df[col] = pd.to_numeric(train_df[col], errors="coerce").astype(float)
    
    for col in config.get("float_to_int_columns", []):
        train_df[col] = train_df[col].apply(lambda x: None if pd.isna(x) else int(x))
    
    print(f"Total records: {len(train_df):,}\n")

    for filename, description, frac, n in datasets_meta:
        filepath = output_dir / filename
        if filepath.exists():
            print(f"  {description:30} -> {filename} (skipped)")
            continue
        if frac is not None:
            df = train_df.sample(frac=frac, random_state=seed)
        elif n is not None:
            df = train_df.sample(n=n, random_state=seed)
        else:
            df = train_df
        df.to_json(filepath, orient="records", indent=2)
        print(f"  {description:30} {len(df):5,} rows -> {filename}")

def load_training_data(config):
    if config["strategy"] not in ["lora"]:
        print("ICL mode: no training data needed")
        return None

    dataset_path = OUTPUT_DIR / Path(config["dataset"]).name
    if not dataset_path.exists():
        raise FileNotFoundError(f"Dataset not found: {dataset_path}")

    train_data = json.loads(dataset_path.read_text(encoding="utf-8"))
    print(f"Training data loaded: {len(train_data)} samples from {dataset_path.name}")
    return train_data

ensure_datasets(OUTPUT_DIR, DATASET_NAME, datasets_config_meta, SEED)
train_data = load_training_data(config)

print("\n===========================| Load Data completed. |===")

## 5. Build Prompt

In [ ]:
print("===========================| Build Prompt started... |===\n")

EXAMPLES_DISCLAIMER = "The following are real records shown only to illustrate the expected format and value ranges.\nYou MUST NOT copy any field values from these examples. Generate completely different data.\n"

def load_prompt(config):
    system_prompt = config["prompt"]["system"]
    user_prompt = config["prompt"]["user"] + "\n"
    if config["strategy"].startswith("few_shot_icl"):
        icl_sample_count = int(config["strategy"].replace("few_shot_icl_", "").replace("r", ""))
        icl_entry = next(
            e for e in config["datasets"]
            if isinstance(e["sample"], int) and e["sample"] == icl_sample_count
        )
        raw_ex = Path(f"{config['dataset_dir']}/{icl_entry['file']}").read_text(encoding="utf-8").strip()
        user_prompt += "\n# EXAMPLES\n" + EXAMPLES_DISCLAIMER + "\n".join(raw_ex.splitlines()[1:-1]).strip() + "\n"
    print(f"System prompt ({len(system_prompt)} chars)")
    print(f"User prompt ({len(user_prompt)} chars)")
    return system_prompt, user_prompt

def build_prompt_tokens(model, tokenizer, system_prompt, user_prompt, config):
    messages = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_prompt},
        {"role": "assistant", "content": config["prompt"]["assistant"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False,
        add_generation_prompt=False,
        continue_final_message=True,
    )
    print("-" * 80)
    print("PROMPT:\n", text)
    print("-" * 80)
    return tokenizer(text, return_tensors="pt").to(next(model.parameters()).device)

system_prompt, user_prompt = load_prompt(config)
prompt_tokens = build_prompt_tokens(model, tokenizer, system_prompt, user_prompt, config)

print("\n===========================| Build Prompt completed. |===")

## 6. Setup Fine-Tuning

In [ ]:
print("===========================| Setup Fine-Tuning started... |===\n")

def setup_peft_model(model, config):
    output_path = f"{config['output_dir']}/{config['name']}_model"
    if config["strategy"] == "lora":
        if Path(output_path).exists():
            print(f"Found existing adapter at '{output_path}', skipping training.")
            config["_skip_training"] = True
            return model
        config["_skip_training"] = False
        peft_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=config["lora_r"],
            lora_alpha=config["lora_alpha"],
            lora_dropout=config["lora_dropout"],
            target_modules="all-linear",
        )
        model = get_peft_model(model, peft_config)
        model.print_trainable_parameters()
    else:
        config["_skip_training"] = True
        print("ICL mode: no fine-tuning needed")
    return model


def tokenize_single(example, system_prompt, user_prompt, tokenizer, assistant_prefix):
    example = dict(example)
    completion = assistant_prefix + json.dumps(example, ensure_ascii=False) + "\n```"
    messages_full = [
        {"role": "system",    "content": system_prompt},
        {"role": "user",      "content": user_prompt},
        {"role": "assistant", "content": completion},
    ]
    messages_prompt = messages_full[:-1]
    full_text   = tokenizer.apply_chat_template(messages_full,   tokenize=False, add_generation_prompt=False)
    prompt_text = tokenizer.apply_chat_template(messages_prompt, tokenize=False, add_generation_prompt=True)
    prompt_text += assistant_prefix
    full_ids   = tokenizer(full_text,   add_special_tokens=False)["input_ids"]
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    prompt_len = len(prompt_ids)
    labels = [-100] * prompt_len + full_ids[prompt_len:]
    return {"input_ids": full_ids, "labels": labels}


class CustomTrainer:
    def __init__(self, model, optimizer, dataloader, config, privacy_engine=None):
        self.model = model
        self.optimizer = optimizer
        self.dataloader = dataloader
        self.config = config
        self.privacy_engine = privacy_engine

    def train(self):
        torch.cuda.empty_cache()
        device = next(self.model.parameters()).device
        self.model.train()
        nan_count = 0
        for epoch in range(self.config["epochs"]):
            total_loss, steps = 0.0, 0
            for batch in tqdm(self.dataloader, desc=f"Epoch {epoch+1}/{self.config['epochs']}"):
                batch = {k: v.to(device) for k, v in batch.items()}
                loss = self.model(**batch).loss
                if not torch.isfinite(loss):
                    print("Nan found")
                    nan_count += 1
                    self.optimizer.zero_grad()
                    continue
                loss.backward()
                if not self.privacy_engine:
                    torch.nn.utils.clip_grad_norm_(
                        filter(lambda p: p.requires_grad, self.model.parameters()),
                        self.config["max_grad_norm"],
                    )
                self.optimizer.step()
                self.optimizer.zero_grad()
                total_loss += loss.item()
                steps += 1
                if steps % 10 == 0:
                    print(f"  step {steps}, loss={total_loss/steps:.4f}")
                if steps % 100 == 0:
                    torch.cuda.empty_cache()
        if nan_count:
            print(f"  Warning: {nan_count} batches NaN/Inf skipped")
        if self.privacy_engine:
            eps = self.privacy_engine.get_epsilon(self.config["dp_delta"])
            print(f"DP training complete — ε = {eps:.4f}, δ = {self.config['dp_delta']}")


def prepare_trainer(model, tokenizer, train_data, config, system_prompt, user_prompt):
    assistant_prefix = config["prompt"]["assistant"]

    for param in model.parameters():
        if param.requires_grad:
            param.data = param.data.to(torch.float32)

    dataset = Dataset.from_list(train_data)
    tokenized = dataset.map(
        lambda ex: tokenize_single(ex, system_prompt, user_prompt, tokenizer, assistant_prefix),
        remove_columns=dataset.column_names,
    )

    torch.cuda.empty_cache()

    collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, padding=True)
    dataloader = DataLoader(
        tokenized,
        batch_size=config["batch_size"],
        collate_fn=collator,
        shuffle=True,
    )
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config["learning_rate"],
        weight_decay=0,
    )

    model.config.use_cache = False
    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": True})

    if config.get("dp", False):
        privacy_engine = PrivacyEngine(accountant=config["dp_accountant"])
        model, optimizer, dataloader = privacy_engine.make_private_with_epsilon(
            module=model,
            optimizer=optimizer,
            data_loader=dataloader,
            epochs=config["epochs"],
            target_epsilon=config["dp_epsilon"],
            target_delta=config["dp_delta"],
            max_grad_norm=config["max_grad_norm"],
            grad_sample_mode=config["dp_grad_sample_mode"],
            alphas=[1 + x / 10.0 for x in range(1, 100)] + list(range(11, 4096)),
            poisson_sampling=False,
        )
        return CustomTrainer(model, optimizer, dataloader, config, privacy_engine)

    return CustomTrainer(model, optimizer, dataloader, config)


set_all_seeds(config["seed"])
model = setup_peft_model(model, config)
trainer = None if config.get("_skip_training") else prepare_trainer(model, tokenizer, train_data, config, system_prompt, user_prompt)

print("\n===========================| Setup Fine-Tuning completed. |===")

## 7. Train

In [ ]:
print("===========================| Train started... |===\n")

def train_model(trainer, model, tokenizer, config):
    output_path = f"{config['output_dir']}/{config['name']}_model"

    if config.get("_skip_training"):
        print(f"Loading pre-trained adapter from '{output_path}'...")
        model = PeftModel.from_pretrained(model, output_path)
        model.eval()
        return model

    if trainer is None:
        print("ICL mode: no training needed")
        return model

    print("Training...")
    trainer.train()
    model.config.use_cache = True
    model.save_pretrained(output_path)
    tokenizer.save_pretrained(output_path)
    print(f"Model saved to: {output_path}")
    return model

model = train_model(trainer, model, tokenizer, config)

print("\n===========================| Train completed. |===")

## 8. Generate Synthetic Data

In [ ]:
print("===========================| Generate Synthetic Data started... |===\n")

TYPE_MAP = {"int": int, "str": str, "float": float}
REQUIRED_FIELDS = {k: TYPE_MAP[v] for k, v in config["required_fields"].items()}
NULLABLE_FIELDS = set(config["nullable_fields"])

def is_valid_record(record):
    if not isinstance(record, dict):
        return False
    if set(record.keys()) != set(REQUIRED_FIELDS.keys()):
        return False
    for field, expected_type in REQUIRED_FIELDS.items():
        value = record.get(field)
        if field in NULLABLE_FIELDS:
            if value is not None and not isinstance(value, expected_type):
                return False
            continue
        if value is None or (isinstance(value, str) and not value.strip()):
            return False
        if not isinstance(value, expected_type):
            if expected_type is float and isinstance(value, int):
                continue
            return False
    return True

def parse_json_response(response):
    try:
        return json.loads(response.strip().split("```")[0].strip())
    except json.JSONDecodeError:
        return None

def generate_samples(model, tokenizer, prompt_tokens, config, output_file):
    model.eval()
    predictions = []
    failed_responses = []
    prompt_token_length = prompt_tokens["input_ids"].shape[1]
    debug_file = output_file.replace(".json", "_debug.json")
    target = config["num_samples"]
    print(f"Generating {target} valid samples...")
    with torch.inference_mode():
        with tqdm(total=target, desc="Generating") as pbar:
            while len(predictions) < target:
                outputs = model.generate(
                    **prompt_tokens,
                    max_new_tokens=config["max_new_tokens"],
                    temperature=config["temperature"],
                    top_p=config["top_p"],
                    do_sample=True,
                    repetition_penalty=config["repetition_penalty"],
                    stop_strings=["```"],
                    tokenizer=tokenizer,
                    pad_token_id=tokenizer.eos_token_id,
                )
                new_tokens = outputs[:, prompt_token_length:]
                response = tokenizer.decode(new_tokens[0], skip_special_tokens=True)
                parsed = parse_json_response(response.strip())
                if parsed is not None and is_valid_record(parsed):
                    predictions.append(parsed)
                    pbar.update(1)
                    with open(output_file, "w", encoding="utf-8") as f:
                        json.dump(predictions, f, indent=2, ensure_ascii=False)
                else:
                    failed_responses.append(response.strip())
                    with open(debug_file, "w", encoding="utf-8") as f:
                        json.dump(failed_responses, f, indent=2, ensure_ascii=False)
    total_attempts = len(predictions) + len(failed_responses)
    print(f"Done: {len(predictions)} valid in {total_attempts} attempts ({len(failed_responses)} discarded)")
    return predictions

output_file = f"{config['output_dir']}/{config['run_name']}.json"
set_all_seeds(config["seed"])
predictions = generate_samples(model, tokenizer, prompt_tokens, config, output_file)
torch.cuda.empty_cache()
print(f"Results saved to: {output_file}")
print("\n===========================| Generate Synthetic Data completed. |===")